# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, overview, and processing of the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined in its Croissant schema at:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

# Optionally, pretty-print other metadata fields
print("\n--- Dataset Metadata Overview ---")
for key in ['keywords', 'spatialCoverage', 'temporalCoverage', 'dataBiases', 'dataCollection', 'dataSocialImpact', 'dataLimitations', 'personalSensitiveInformation']:
    val = getattr(dataset.metadata, key, None)
    if val:
        print(f"{key}: {val}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.
All entities are referenced by their `@id`. We'll first discover the record sets in the dataset.

In [ ]:
# List all RecordSets and their IDs

record_sets = dataset.metadata.recordSet
print("Record Sets:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '(Unnamed)')}")

# For each record set, list its fields and columns
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name')})")
    fields = rs.get('field', [])
    columns = rs.get('column', [])
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    - {f['@id']} ({f.get('name', '')})")
    if columns:
        print("  Columns:")
        for c in columns:
            print(f"    - {c['@id']} ({c.get('name', '')})")

## 3. Data Extraction
Load data from specific record sets using their `@id`. All extraction references entities by `@id`, following the Croissant schema.

In [ ]:
# Collect record sets by their @id

record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
print("Extracting DataFrames for record sets:")
pprint.pprint(record_set_ids)

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    # Store as DataFrame
    dataframes[record_set_id] = pd.DataFrame(records)

# Pick the first record set for demonstration
if record_set_ids:
    main_rs = record_set_ids[0]
    print(f"Available columns in {main_rs}:")
    print(dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
Perform filtering, normalization, and grouping on relevant fields. All references use the `@id` values.
We'll assume a column with numeric values is present, such as log likelihood or coefficient. Adapt as needed per the output from the previous cell.

In [ ]:
# Identify numeric fields (example: 'log_likelihood', 'coefficient', or similar)

# You may need to adapt this field name to match the actual schema
example_numeric_field_id = None

# Search for numeric columns in the first record set
df = dataframes[main_rs]
# Try to get a numeric field (@id, typically under columns)
for col in df.columns:
    # Try matching typical numeric fields
    if 'log_likelihood' in col or 'coefficient' in col or 'standard_error' in col or 'p_value' in col:
        example_numeric_field_id = col
        break
if example_numeric_field_id is None:
    # Default to first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            example_numeric_field_id = col
            break

print(f"Using numeric field: {example_numeric_field_id}")

# Filter records for numeric_field > threshold
threshold = 10
filtered_df = df[df[example_numeric_field_id] > threshold]
print(f"Filtered records where {example_numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
norm_col = f"{example_numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) / filtered_df[example_numeric_field_id].std()
print(f"Normalized {example_numeric_field_id} for filtered records:")
display(filtered_df[[example_numeric_field_id, norm_col]].head())

# Group by a categorical field if available
group_field_id = None
# Try to find a string/object column
for col in df.columns:
    if pd.api.types.is_object_dtype(df[col]) and col != example_numeric_field_id:
        group_field_id = col
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No suitable categorical grouping field found.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset. We'll use matplotlib for simple plots.

In [ ]:
# Basic visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[example_numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {example_numeric_field_id}")
plt.xlabel(example_numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If grouping field found, show boxplot
if group_field_id:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field_id, y=example_numeric_field_id)
    plt.title(f"{example_numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated:
- Loading and overview of a FAIR² dataset via Croissant schema.
- Data extraction referenced by `@id`.
- Exploratory filtering and normalization of numeric fields.
- Data grouping by categorical fields.
- Simple visualizations of distributions and group statistics.

Further work can include deeper statistical analysis or machine learning tasks, all leveraging transparent and interoperable Croissant schema access via `mlcroissant`.